# 🎬 Hindi Video Dubbing Pipeline
## Supernan AI Intern Challenge — "The Golden 15 Seconds"

This notebook runs the full pipeline end-to-end on Google Colab (Free T4 GPU).

**Pipeline:**
1. Extract 15-second segment from source video
2. Transcribe English audio (Whisper)
3. Translate to Hindi (MarianMT)
4. Synthesize Hindi voice (Coqui XTTS v2 — voice cloning)
5. Lip-sync video (Wav2Lip)
6. Enhance face quality (GFPGAN)

**Cost: ₹0** — All models are free and open-source.

## Step 0: Check GPU & Install Dependencies

In [ ]:
# Verify GPU availability
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Pipeline will run on CPU (slow for lip-sync).')

In [ ]:
# Install system dependencies
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ ffmpeg installed')

In [ ]:
# Install Python dependencies (this takes ~3-5 minutes on first run)
!pip install -q openai-whisper
!pip install -q transformers sentencepiece deep-translator
!pip install -q TTS gTTS
!pip install -q librosa soundfile pydub
!pip install -q opencv-python gfpgan facexlib basicsr realesrgan
!pip install -q gdown ffmpeg-python
print('✅ All dependencies installed')

In [ ]:
# Clone the project repository
import os
if not os.path.exists('Supernan_video_dubbing_project'):
    !git clone https://github.com/Harshrj53/Supernan_video_dubbing_project.git
%cd Supernan_video_dubbing_project/ai_voice_translater
print(f'Working directory: {os.getcwd()}')

## Step 1: Upload Source Video

Upload the Supernan training video downloaded from Google Drive.
Or mount Google Drive if you've saved the video there.

In [ ]:
# Option A: Upload from local machine
from google.colab import files
print('Please upload your video file (MP4)...')
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {video_filename}')

In [ ]:
# Option B: Mount Google Drive (if video is stored in Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# video_filename = '/content/drive/MyDrive/supernan_video.mp4'

## Step 2: Configure Pipeline Parameters

In [ ]:
import os
import torch

# ── Pipeline Configuration ────────────────────────────────────────────────
INPUT_VIDEO    = video_filename   # Source video
OUTPUT_VIDEO   = 'output_hindi_dubbed.mp4'  # Final output
START_SEC      = 15.0            # Segment start time (seconds)
END_SEC        = 30.0            # Segment end time (seconds)

# Model selection
WHISPER_MODEL  = 'base'          # 'base' for speed; 'large-v3' for accuracy
LIPSYNC_MODEL  = 'wav2lip'       # 'wav2lip' or 'videoretalking'
ENHANCE_MODEL  = 'gfpgan'        # 'gfpgan', 'codeformer', or 'none'

# Compute
USE_GPU        = torch.cuda.is_available()
ADD_SUBTITLES  = True            # Burn Hindi subtitles into output

print(f'Input     : {INPUT_VIDEO}')
print(f'Segment   : {START_SEC}s → {END_SEC}s ({END_SEC-START_SEC}s)')
print(f'Whisper   : {WHISPER_MODEL}')
print(f'Lip-sync  : {LIPSYNC_MODEL}')
print(f'Enhance   : {ENHANCE_MODEL}')
print(f'GPU       : {USE_GPU}')

## Step 3: Run the Full Pipeline

In [ ]:
# Run pipeline via CLI (cleanest way — all logic in dub_video.py)
cmd = (
    f'python dub_video.py '
    f'--input {INPUT_VIDEO} '
    f'--start {START_SEC} '
    f'--end {END_SEC} '
    f'--output {OUTPUT_VIDEO} '
    f'--whisper-model {WHISPER_MODEL} '
    f'--lipsync-model {LIPSYNC_MODEL} '
    f'--enhance-model {ENHANCE_MODEL} '
    + ('--gpu ' if USE_GPU else '') 
    + ('--add-subtitles' if ADD_SUBTITLES else '')
)
print(f'Running: {cmd}')
!{cmd}

## Step 4: Preview Output

In [ ]:
from IPython.display import Video, HTML
import os

if os.path.exists(OUTPUT_VIDEO):
    size_mb = os.path.getsize(OUTPUT_VIDEO) / 1e6
    print(f'✅ Output video: {OUTPUT_VIDEO} ({size_mb:.1f} MB)')
    display(Video(OUTPUT_VIDEO, width=640, embed=True))
else:
    print('❌ Output video not found — check pipeline logs above')

## Step 5: Download Output

In [ ]:
from google.colab import files
files.download(OUTPUT_VIDEO)

---
## 🔬 Optional: Run Steps Individually (for Debugging)

In [ ]:
# Step 1 only: Extract 15s clip
from modules.extract import extract_segment

paths = extract_segment(
    video_path=INPUT_VIDEO,
    start_sec=START_SEC,
    end_sec=END_SEC,
    output_dir='./debug_output/01_extract',
)
print('Extracted:', paths)

In [ ]:
# Step 2 only: Whisper transcription
from modules.transcribe import transcribe_audio

transcript = transcribe_audio(
    audio_path=paths['audio'],
    model_size=WHISPER_MODEL,
    output_dir='./debug_output/02_transcribe',
)
print('Full English text:')
print(transcript.full_text)
print(f'\n{len(transcript.segments)} segments:')
for seg in transcript.segments:
    print(f'  [{seg.start:.1f}s→{seg.end:.1f}s] {seg.text}')

In [ ]:
# Step 3 only: Hindi translation
from modules.translate import translate_to_hindi

hindi_full, translated_segs = translate_to_hindi(transcript.segments)

print('Hindi translation:')
print(hindi_full)
print('\nSegment-by-segment:')
for ts in translated_segs:
    print(f'  EN: {ts.english}')
    print(f'  HI: {ts.hindi}\n')

In [ ]:
# Step 4 only: TTS with voice cloning
from modules.tts import synthesize_hindi_voice

hindi_audio = synthesize_hindi_voice(
    hindi_text=hindi_full,
    reference_audio=paths['ref_audio'],
    output_path='./debug_output/04_tts/hindi_dubbed.wav',
    target_duration_sec=(END_SEC - START_SEC),
    use_gpu=USE_GPU,
)
print(f'Hindi audio: {hindi_audio}')

# Play it
import IPython.display as ipd
ipd.display(ipd.Audio(hindi_audio))

In [ ]:
# Step 5 only: Wav2Lip lip-sync
from modules.lipsync import run_lipsync

lipsynced = run_lipsync(
    video_path=paths['video'],
    audio_path=hindi_audio,
    output_path='./debug_output/05_lipsync/lipsynced.mp4',
    use_gpu=USE_GPU,
)
print(f'Lip-synced: {lipsynced}')
from IPython.display import Video
display(Video(lipsynced, width=640, embed=True))

In [ ]:
# Step 6 only: GFPGAN face enhancement
from modules.enhance import enhance_video

enhanced = enhance_video(
    video_path=lipsynced,
    output_path='./debug_output/06_enhance/enhanced.mp4',
    model='gfpgan',
)
print(f'Enhanced: {enhanced}')
display(Video(enhanced, width=640, embed=True))

---
## 📊 Scaling to 500 Hours Overnight

```python
from multiprocessing import Pool
import subprocess

def process_segment(args):
    video, start, end, output = args
    subprocess.run([
        'python', 'dub_video.py',
        '--input', video, '--start', str(start),
        '--end', str(end), '--output', output, '--gpu'
    ])

# Generate segment list: 15s chunks of a 500h video
segments = [
    ('video.mp4', i, i+15, f'output_{i}.mp4')
    for i in range(0, 500*3600, 15)
]

# Process in parallel across 8 GPU workers
with Pool(processes=8) as pool:
    pool.map(process_segment, segments)
```